# 02 · 모델 실험 (전처리 → 분리 → 학습 → 평가 → 저장)

**규율**: ①먼저 분리 ②전처리기는 Train에만 fit ③모델·임계값은 Validation에서 ④Test는 한 번 ⑤Pipeline 통째 저장 ⑥누수 컬럼 제거

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from src.predict import load_config, resolve_path
from src.clean import clean_raw  # 공통 정제 규칙 (A 관리)

cfg = load_config(); SEED = cfg['project']['random_state']
df = clean_raw(pd.read_csv(resolve_path(cfg['data']['raw_path']), encoding=cfg['data']['encoding'], sep=cfg['data']['sep']))
tcol = cfg['target']['column']; pos = str(cfg['target']['positive_label']).strip().lower()
y = df[tcol].astype(str).str.strip().str.lower().eq(pos).astype(int)

id_cols = []        # TODO 예: ['customerID']
leakage_cols = []   # TODO 예: ['해지일','이탈사유']
X = df.drop(columns=[tcol] + id_cols + leakage_cols)
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

## 1) 분리 (먼저!) + 2) 전처리 Pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=SEED)

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

## 3) 모델 비교 (동일 split·동일 지표)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, average_precision_score

models = {'Dummy': DummyClassifier(strategy='most_frequent'),
          'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced'),
          'RF': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1)}
# TODO: XGBoost/LightGBM 등 부스팅 1개 추가

rows = {}
for name, est in models.items():
    pipe = Pipeline([('prep', prep), ('model', est)]); pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_val)[:, 1]; pred = p >= 0.5
    rows[name] = {'recall': recall_score(y_val, pred), 'precision': precision_score(y_val, pred, zero_division=0),
                  'f1': f1_score(y_val, pred), 'pr_auc': average_precision_score(y_val, p)}
pd.DataFrame(rows).T.round(4)

## 4) 최종 모델 선정(근거!) + 임계값(Validation)

In [ ]:
best = Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])  # TODO 교체
best.fit(X_train, y_train)
val_p = best.predict_proba(X_val)[:, 1]
grid = np.linspace(0.05, 0.95, 91)
thr = float(max(grid, key=lambda t: f1_score(y_val, val_p >= t)))
print('threshold =', round(thr, 2))

## 5) 최종 Test (한 번)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
test_p = best.predict_proba(X_test)[:, 1]; test_pred = (test_p >= thr).astype(int)
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, digits=4))

## 6) 저장 (Streamlit이 쓰는 형식)

In [ ]:
import joblib, json
model_path = resolve_path(cfg['output']['model_path']); model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({'pipeline': best, 'threshold': thr, 'feature_names': list(X.columns)}, model_path)
pd.DataFrame(rows).T.round(4).to_csv(resolve_path(cfg['output']['metrics_path']))  # 성능 탭용
print('저장 완료:', model_path)